# Notebook 02 — Simple Token Subgroup Baseline

This notebook trains the **simple token conditioning** model.

The subgroup is inserted directly into the RoBERTa input sequence as plain text:

```text
### COMMENT TO CLASSIFY
<comment>

### ANNOTATOR IDEOLOGY / IDENTITY
<subgroup>
```

Architecture:

```text
comment + subgroup text
        ↓
RoBERTa-base
        ↓
CLS representation
        ↓
linear classifier
        ↓
3-class probability distribution
```

No subgroup embedding.  
No FiLM.  
No retrieved context.

The notebook is controlled by `DISCOURSE`, so the same code can later be reused for `immigration` or `women`.


## 1. Imports and Configuration

In [49]:
import os
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    confusion_matrix,
    classification_report,
)
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

# ---------------------------------------------------------------------
# Main switch. Change this to "women" for the women-targeted discourse.
# ---------------------------------------------------------------------
DISCOURSE = "women"  # options: "immigration", "women"

DISCOURSE_CONFIG = {
    "immigration": {
        "processed_path": "immigration_processed.parquet",
        "subgroup_header": "ANNOTATOR IDEOLOGY",
        "allowed_subgroups": ["liberal", "conservative","extremely_liberal", "extremely_conservative", "slightly_liberal", "slightly_conservative", "no_opinion", "neutral"],
        "output_dir": "immigration_simple_token_outputs",
        "output_prefix": "immigration_simple_token",
    },
    "women": {
        "processed_path": "women_processed (1).parquet",
        "subgroup_header": "ANNOTATOR GENDER IDENTITY",
        # Leave as None to use every subgroup present in the processed file.
        "allowed_subgroups": ["women", "men", "non_binary"],
        "output_dir": "women_simple_token_outputs",
        "output_prefix": "women_simple_token",
    },
}

assert DISCOURSE in DISCOURSE_CONFIG, f"Unknown discourse: {DISCOURSE}"
CFG = DISCOURSE_CONFIG[DISCOURSE]

RANDOM_SEED = 42
MODEL_NAME = "roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 192
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data")
OUTPUT_DIR = Path(CFG["output_dir"])
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

EXPERIMENT = DISCOURSE
OUTPUT_PREFIX = CFG["output_prefix"]

print("Discourse:", DISCOURSE)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR.resolve())


Discourse: women
Device: cuda
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_simple_token_outputs


In [50]:
def set_seed(seed: int = 42) -> None:
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Load Processed Data

This expects Notebook 1 to have created the processed file selected by `DISCOURSE`:

- `immigration_processed.parquet`
- `women_processed.parquet`


In [51]:
data_path = DATA_DIR / CFG["processed_path"]

if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {data_path}. Make sure Notebook 1 created the processed file."
    )

comment_df = pd.read_parquet(data_path)

print("Loaded:", data_path)
print("Shape:", comment_df.shape)
display(comment_df.head(2))


Loaded: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data/women_processed (1).parquet
Shape: (3953, 12)


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,6,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),train,2,"[2.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0.0,0,0.0,"{'men': 1.0, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1.0}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}"
1,11,"eat my fuck, bitch",validation,2,"[0.0, 1.0, 1.0]","[0.0, 0.5, 0.5]",1.0,1,1.5,"{'men': 1.0, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1.0}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}"


In [52]:
required_columns = {
    "comment_id",
    "text",
    "split",
    "subgroup_distributions",
    "subgroup_counts",
}

missing = required_columns - set(comment_df.columns)
assert not missing, f"Dataset is missing columns: {missing}"

print("Required columns are present.")
print("Split counts:")
display(comment_df["split"].value_counts())


Required columns are present.
Split counts:


split
train         2763
validation     608
test           582
Name: count, dtype: int64

## 3. Helper Functions

In [53]:
def ensure_dict(value):
    """Return a dictionary whether the stored value is already a dict or a stringified dict."""
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    raise TypeError(f"Expected dict or stringified dict, got {type(value)}")


def is_valid_distribution(dist, num_labels: int = NUM_LABELS, tolerance: float = 1e-5) -> bool:
    """Check that a distribution has the right length and sums approximately to one."""
    if len(dist) != num_labels:
        return False
    if any(float(p) < -tolerance for p in dist):
        return False
    return abs(sum(float(p) for p in dist) - 1.0) < tolerance


## 4. Expand Comment-Level Data into Subgroup-Level Examples

Notebook 1 gives one row per comment.

Notebook 2 needs one row per **comment-subgroup pair**.


In [54]:
def build_simple_token_input(text: str, subgroup: str, subgroup_header: str) -> str:
    """Build the simple token conditioning input string.

    The subgroup is plain text inside the RoBERTa sequence.
    There is no context and no separate subgroup embedding.
    """
    return (f"### COMMENT TO CLASSIFY{text}"f"### {subgroup_header}{subgroup}")


In [55]:
def expand_subgroup_examples(
    comment_df: pd.DataFrame,
    experiment_name: str,
    subgroup_header: str,
    allowed_subgroups: list[str] | None = None,
) -> pd.DataFrame:

    rows = []
    skipped_none = 0
    skipped_invalid = 0
    skipped_not_allowed = 0

    for _, row in comment_df.iterrows():
        subgroup_distributions = ensure_dict(row["subgroup_distributions"])
        subgroup_counts = ensure_dict(row["subgroup_counts"])

        for subgroup, target_distribution in subgroup_distributions.items():

            if allowed_subgroups is not None and subgroup not in allowed_subgroups:
                skipped_not_allowed += 1
                continue

            if target_distribution is None:
                skipped_none += 1
                continue

            target_distribution = [float(x) for x in target_distribution]

            if not is_valid_distribution(target_distribution):
                skipped_invalid += 1
                continue

            rows.append({
                "experiment": experiment_name,
                "comment_id": row["comment_id"],
                "split": row["split"],
                "subgroup": subgroup,
                "subgroup_count": int(subgroup_counts.get(subgroup, 0)),
                "text": row["text"],
                "input_text": build_simple_token_input(row["text"], subgroup, subgroup_header),
                "target_distribution": target_distribution,
                "target_majority_label": int(np.argmax(target_distribution)),
                "target_expected_label": float(np.dot(np.arange(NUM_LABELS), target_distribution)),
                "target_entropy": float(entropy(target_distribution, base=2)),
            })

    expanded_df = pd.DataFrame(rows)

    print(f"{experiment_name}: skipped {skipped_none} None distributions")
    print(f"{experiment_name}: skipped {skipped_invalid} invalid distributions")
    print(f"{experiment_name}: skipped {skipped_not_allowed} not-allowed subgroups")
    print(f"{experiment_name}: created {len(expanded_df):,} subgroup examples")

    return expanded_df


In [56]:
model_examples = expand_subgroup_examples(
    comment_df=comment_df,
    experiment_name=DISCOURSE,
    subgroup_header=CFG["subgroup_header"],
    allowed_subgroups=CFG["allowed_subgroups"],
)

print("Subgroup examples:", model_examples.shape)
display(model_examples.head())

print("Example input text:")
print(model_examples.iloc[0]["input_text"][:1000])


women: skipped 3897 None distributions
women: skipped 0 invalid distributions
women: skipped 7906 not-allowed subgroups
women: created 7,962 subgroup examples
Subgroup examples: (7962, 11)


,experiment,comment_id,split,subgroup,subgroup_count,text,input_text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),### COMMENT TO CLASSIFYFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)### ANNOTATOR GENDER IDENTITYmen,"[1.0, 0.0, 0.0]",0,0.0,0.0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),### COMMENT TO CLASSIFYFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)### ANNOTATOR GENDER IDENTITYwomen,"[1.0, 0.0, 0.0]",0,0.0,0.0
2,women,11,validation,men,1,"eat my fuck, bitch","### COMMENT TO CLASSIFYeat my fuck, bitch### ANNOTATOR GENDER IDENTITYmen","[0.0, 0.0, 1.0]",2,2.0,0.0
3,women,11,validation,women,1,"eat my fuck, bitch","### COMMENT TO CLASSIFYeat my fuck, bitch### ANNOTATOR GENDER IDENTITYwomen","[0.0, 1.0, 0.0]",1,1.0,0.0
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,### COMMENT TO CLASSIFYI'd love to rip those panties off and shove my fat cock in her ass.### ANNOTATOR GENDER IDENTITYmen,"[0.5, 0.0, 0.5]",0,1.0,1.0


Example input text:
### COMMENT TO CLASSIFYFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)### ANNOTATOR GENDER IDENTITYmen


In [57]:
print("Split × subgroup counts")
display(pd.crosstab(model_examples["split"], model_examples["subgroup"]))


Split × subgroup counts


subgroup,men,non_binary,women
split,,,
test,576,16,581
train,2723,108,2734
validation,604,16,604


## 5. Select Model Data

In [58]:
model_df = model_examples.copy()

print(model_df.shape)
display(model_df.head())


(7962, 11)


,experiment,comment_id,split,subgroup,subgroup_count,text,input_text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),### COMMENT TO CLASSIFYFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)### ANNOTATOR GENDER IDENTITYmen,"[1.0, 0.0, 0.0]",0,0.0,0.0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),### COMMENT TO CLASSIFYFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)### ANNOTATOR GENDER IDENTITYwomen,"[1.0, 0.0, 0.0]",0,0.0,0.0
2,women,11,validation,men,1,"eat my fuck, bitch","### COMMENT TO CLASSIFYeat my fuck, bitch### ANNOTATOR GENDER IDENTITYmen","[0.0, 0.0, 1.0]",2,2.0,0.0
3,women,11,validation,women,1,"eat my fuck, bitch","### COMMENT TO CLASSIFYeat my fuck, bitch### ANNOTATOR GENDER IDENTITYwomen","[0.0, 1.0, 0.0]",1,1.0,0.0
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,### COMMENT TO CLASSIFYI'd love to rip those panties off and shove my fat cock in her ass.### ANNOTATOR GENDER IDENTITYmen,"[0.5, 0.0, 0.5]",0,1.0,1.0


In [59]:
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0, "Training set is empty."
assert len(val_df) > 0, "Validation set is empty."
assert len(test_df) > 0, "Test set is empty."


Train: (5565, 11)
Val: (1224, 11)
Test: (1173, 11)


## 6. Tokenization and Dataset Class

In [60]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [61]:
class HateSpeechDistributionDataset(Dataset):
    """PyTorch dataset for subgroup-conditioned distribution prediction."""

    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int = 192):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["input_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "target_distribution": torch.tensor(row["target_distribution"], dtype=torch.float),
        }


In [62]:
train_dataset = HateSpeechDistributionDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = HateSpeechDistributionDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = HateSpeechDistributionDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([16, 192]),
 'attention_mask': torch.Size([16, 192]),
 'target_distribution': torch.Size([16, 3])}

## 7. Model

The model returns log-probabilities because `nn.KLDivLoss` expects log-probabilities as input.


In [63]:
class SimpleTokenRoBERTa(nn.Module):
    """RoBERTa distribution predictor with subgroup represented as plain input text."""

    def __init__(self, model_name: str, num_labels: int = 3, dropout: float = 0.1):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_embedding))

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


In [64]:
model = SimpleTokenRoBERTa(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)
print("RoBERTa layers:", model.encoder.config.num_hidden_layers)
print("Hidden size:", model.encoder.config.hidden_size)
print("Attention heads:", model.encoder.config.num_attention_heads)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training steps: 3480
Warmup steps: 348
RoBERTa layers: 12
Hidden size: 768
Attention heads: 12


## 8. Metric Functions

In [65]:
EPS = 1e-12


def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """KL(y_true || y_pred) per row."""
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)


def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Jensen-Shannon divergence per row."""
    values = []
    for true_dist, pred_dist in zip(y_true, y_pred):
        values.append(jensenshannon(true_dist, pred_dist, base=2) ** 2)
    return np.array(values)


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Cross-entropy H(y_true, y_pred) per row."""
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)


def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels


def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro")),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## 9. Training and Evaluation Functions

In [66]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None

    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs["log_probs"], targets)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


## 10. Train Baseline

In [67]:
best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_best_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scheduler=None,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)


Epoch 1/10
Train KL: 0.7200 | Val KL: 0.6514
Train JS: 0.2856 | Val JS: 0.2434
Val Acc: 0.7198 | Val Macro F1: 0.4736
Saved new best model to women_simple_token_outputs/women_simple_token_best_model.pt
Epoch 2/10
Train KL: 0.6160 | Val KL: 0.6230
Train JS: 0.2348 | Val JS: 0.2472
Val Acc: 0.7222 | Val Macro F1: 0.4781
Saved new best model to women_simple_token_outputs/women_simple_token_best_model.pt
Epoch 3/10
Train KL: 0.5743 | Val KL: 0.6461
Train JS: 0.2188 | Val JS: 0.2396
Val Acc: 0.7214 | Val Macro F1: 0.4760
Epoch 4/10
Train KL: 0.5375 | Val KL: 0.6451
Train JS: 0.2065 | Val JS: 0.2354
Val Acc: 0.7141 | Val Macro F1: 0.4779
Epoch 5/10
Train KL: 0.5063 | Val KL: 0.6616
Train JS: 0.1941 | Val JS: 0.2354
Val Acc: 0.7255 | Val Macro F1: 0.4796
Epoch 6/10
Train KL: 0.4700 | Val KL: 0.7539
Train JS: 0.1820 | Val JS: 0.2357
Val Acc: 0.7149 | Val Macro F1: 0.4771
Epoch 7/10
Train KL: 0.4290 | Val KL: 0.8074
Train JS: 0.1679 | Val JS: 0.2330
Val Acc: 0.7190 | Val Macro F1: 0.4708
Epoch 

,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.719966,0.285595,0.800573,0.693980,0.396919,0.665687,0.078784,0.065046,0.719966,0.651367,0.243424,0.724497,0.719771,0.473620,0.529667,0.096024,0.084210,0.651367
1,2,0.615967,0.234841,0.696573,0.720934,0.464949,0.532364,0.149752,0.146942,0.615967,0.622991,0.247185,0.696121,0.722222,0.478145,0.540242,0.135509,0.122131,0.622991
2,3,0.574316,0.218750,0.654922,0.742677,0.486455,0.488235,0.167804,0.157534,0.574316,0.646091,0.239642,0.719220,0.721405,0.475965,0.523629,0.132815,0.132269,0.646091
3,4,0.537476,0.206457,0.618082,0.751482,0.495004,0.457731,0.205699,0.192502,0.537476,0.645095,0.235426,0.718225,0.714052,0.477928,0.521962,0.131564,0.123847,0.645095
4,5,0.506349,0.194100,0.586955,0.762264,0.504976,0.428696,0.214415,0.205919,0.506349,0.661576,0.235352,0.734706,0.725490,0.479620,0.514766,0.134411,0.127345,0.661576
5,6,0.470013,0.182042,0.550620,0.775921,0.528167,0.401332,0.235730,0.224531,0.470013,0.753913,0.235694,0.827043,0.714869,0.477112,0.498099,0.110960,0.115465,0.753913
6,7,0.429032,0.167938,0.509638,0.789757,0.580759,0.373820,0.257482,0.248080,0.429032,0.807400,0.232953,0.880530,0.718954,0.470787,0.491135,0.108126,0.113564,0.807400
7,8,0.401750,0.157247,0.482356,0.801617,0.621621,0.353823,0.267124,0.256922,0.401750,0.813159,0.238298,0.886289,0.711601,0.466649,0.493006,0.112303,0.114856,0.813159
8,9,0.376891,0.149419,0.457498,0.805391,0.653955,0.341069,0.272880,0.261993,0.376891,0.804407,0.237600,0.877537,0.710784,0.476666,0.501449,0.102754,0.106397,0.804407
9,10,0.346896,0.138968,0.427502,0.819766,0.674494,0.321961,0.289383,0.277842,0.346896,0.834275,0.237715,0.907405,0.705882,0.472291,0.495116,0.101340,0.107770,0.834275


In [68]:
history_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Saved: women_simple_token_outputs/women_simple_token_training_history.csv


## 11. Test Evaluation

In [69]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scheduler=None,
)

test_metrics


/tmp/ipykernel_1753/3949287068.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.665467381477356,
 'js_mean': 0.25929393369705667,
 'cross_entropy_mean': 0.7489305138587952,
 'accuracy': 0.6947996589940324,
 'macro_f1': 0.45407030867074444,
 'expected_label_mae': 0.5824274023985276,
 'entropy_pearson': 0.11341402269979772,
 'entropy_spearman': 0.11313001546716377,
 'loss': 0.6654674311351898}

In [70]:
metrics_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:", metrics_path)


Saved: women_simple_token_outputs/women_simple_token_test_metrics.json


## 12. Save Test Predictions

In [71]:
predictions_df = test_df.copy()

predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)

predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)

predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

display(predictions_df.head())


,experiment,comment_id,split,subgroup,subgroup_count,text,input_text,target_distribution,target_majority_label,target_expected_label,target_entropy,true_distribution,pred_distribution,pred_majority_label,pred_expected_label,pred_entropy,kl,js,cross_entropy
0,women,63,test,men,1,Demand she pay and topple the patriarchy!,### COMMENT TO CLASSIFYDemand she pay and topple the patriarchy!### ANNOTATOR GENDER IDENTITYmen,"[1.0, 0.0, 0.0]",0,0.0,0.0,"[1.0, 0.0, 0.0]","[0.85397226, 0.07552833, 0.07049939]",0,0.216527,0.745713,0.157857,0.077167,0.157857
1,women,63,test,women,2,Demand she pay and topple the patriarchy!,### COMMENT TO CLASSIFYDemand she pay and topple the patriarchy!### ANNOTATOR GENDER IDENTITYwomen,"[1.0, 0.0, 0.0]",0,0.0,0.0,"[1.0, 0.0, 0.0]","[0.8548204, 0.075262584, 0.06991708]",0,0.215097,0.742682,0.156864,0.076693,0.156864
2,women,193,test,men,1,I will personally rape your mother,### COMMENT TO CLASSIFYI will personally rape your mother### ANNOTATOR GENDER IDENTITYmen,"[0.0, 0.0, 1.0]",2,2.0,0.0,"[0.0, 0.0, 1.0]","[0.7673544, 0.1107843, 0.12186125]",0,0.354507,1.014860,2.104872,0.721918,2.104872
3,women,193,test,women,1,I will personally rape your mother,### COMMENT TO CLASSIFYI will personally rape your mother### ANNOTATOR GENDER IDENTITYwomen,"[0.0, 0.0, 1.0]",2,2.0,0.0,"[0.0, 0.0, 1.0]","[0.7646615, 0.1114712, 0.12386727]",0,0.359206,1.022069,2.088545,0.718716,2.088545
4,women,209,test,men,1,I hope she's been sterilized,### COMMENT TO CLASSIFYI hope she's been sterilized### ANNOTATOR GENDER IDENTITYmen,"[1.0, 0.0, 0.0]",0,0.0,0.0,"[1.0, 0.0, 0.0]","[0.7836237, 0.11109778, 0.10527851]",0,0.321655,0.969759,0.243826,0.117679,0.243826


In [72]:
predictions_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

print("Saved:", predictions_path)


predictions_csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_test_predictions.csv"
predictions_df.to_csv(predictions_csv_path, index=False)
print("Saved:", predictions_csv_path)


Saved: women_simple_token_outputs/women_simple_token_test_predictions.parquet
Saved: women_simple_token_outputs/women_simple_token_test_predictions.csv


## 13. Basic Error Analysis

In [73]:
print("Worst examples by KL divergence")

display(
    predictions_df
    .sort_values("kl", ascending=False)
    [[
        "comment_id",
        "subgroup",
        "text",
        "target_distribution",
        "pred_distribution",
        "kl",
        "js",
    ]]
    .head(10)
)


Worst examples by KL divergence


,comment_id,subgroup,text,target_distribution,pred_distribution,kl,js
355,11994,women,Indian women deserve all the rape and female infanticide they get from Muslims in India,"[0.0, 0.0, 1.0]","[0.9517028, 0.030321052, 0.017976193]",4.018707,0.934806
632,23725,men,Females are fucking stupid,"[0.0, 0.0, 1.0]","[0.93805987, 0.03505984, 0.02688026]",3.616363,0.910230
633,23725,women,Females are fucking stupid,"[0.0, 0.0, 1.0]","[0.9376651, 0.035284184, 0.027050678]",3.610043,0.909783
207,6660,women,I know this is neither relevant to running or transition but reading your post just gave me insight to myself. I've got a medical disorder that causes me to have high testosterone for a woman. I've had it all my life...,"[0.0, 0.0, 1.0]","[0.9297884, 0.042361174, 0.02785038]",3.580909,0.907693
1110,45871,women,Feminists are female incels,"[0.0, 0.5, 0.5]","[0.96831346, 0.021493243, 0.010193333]",3.519872,0.899327
71,2389,men,I like the idea of killing babies but on the other hand I dont want women to have a choice.,"[0.0, 0.0, 1.0]","[0.92384684, 0.04379184, 0.03236124]",3.430794,0.896195
317,10760,women,You hate women?,"[0.0, 1.0, 0.0]","[0.94473445, 0.032908622, 0.022356937]",3.414021,0.894831
405,13920,women,"God, women and their herd nature are insufferable. There's a reason they were not considered true citizens with full rights for most of human history...","[0.0, 0.0, 1.0]","[0.9131402, 0.048029255, 0.038830537]",3.248548,0.880460
825,31848,men,Women's are trash,"[0.0, 0.0, 1.0]","[0.9134033, 0.046665277, 0.039931506]",3.220590,0.877861
188,5983,men,"Feminists: ""B-b-b-b-but sex abuse is a \~male\~ crime !!! Wimmenz are sugar and spice and everything nice ! Ever since the beginning of time, women have been pure, good, and angelic ! Wimmenz cannot commit such offen...","[0.0, 0.0, 1.0]","[0.90086114, 0.05843685, 0.040702075]",3.201476,0.876054


In [74]:
print("Performance by subgroup")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }

    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")

display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)

print("Saved:", subgroup_metrics_path)


Performance by subgroup


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,non_binary,16,0.532449,0.207044,0.532449,0.812500,0.539855,0.455221,NaN,NaN
0,men,576,0.653980,0.256205,0.746574,0.678819,0.432856,0.580667,0.145559,0.152378
2,women,581,0.680519,0.263796,0.757228,0.707401,0.471254,0.587675,0.075254,0.067274


Saved: women_simple_token_outputs/women_simple_token_subgroup_metrics.csv


from sklearn.metrics import confusion_matrix, classification_report

true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())

print("\nAverage target distribution:")
print(np.vstack(predictions_df["true_distribution"].to_numpy()).mean(axis=0))

print("\nAverage predicted distribution:")
print(np.vstack(predictions_df["pred_distribution"].to_numpy()).mean(axis=0))


## 14. Notes for Unified Evaluation

This simple token notebook saves standardized files into `OUTPUT_DIR`:

- `*_best_model.pt`
- `*_training_history.csv`
- `*_test_metrics.json`
- `*_test_predictions.parquet`
- `*_test_predictions.csv`
- `*_subgroup_metrics.csv`

These can be loaded by the unified evaluation notebook and compared against Original A, Embedding, FiLM, and context variants.
